# Two-Line Element Set

Two-Line Element sets (TLEs) are the most widely used format for distributing satellite orbital data. They encode mean orbital elements designed specifically for use with the SGP4 propagator, and are published by organizations like [CelesTrak](https://celestrak.org) and [Space-Track](https://www.space-track.org).

This tutorial demonstrates:

1. Loading a TLE and propagating it with SGP4 to get a position and velocity in the **TEME** (True Equator Mean Equinox) frame
2. Rotating from TEME to the **ITRF** (Earth-fixed) frame to obtain geodetic coordinates
3. Plotting the satellite ground track over multiple orbits

## Generate State Vector

SGP4 outputs position and velocity in the TEME frame, a pseudo-inertial frame that does not account for precession or nutation. To get a location on the Earth's surface, we rotate into the ITRF frame using a quaternion from `satkit.frametransform`, then convert the Cartesian position to geodetic coordinates.

In [ ]:
import satkit as sk

# The two-line element set
# Lets pick a random StarLink satellite
# The lines below were downloaded from https://www.celetrack.org
tle_lines = [
    '0 STARLINK-30477',
    '1 57912U 23146X   24099.49439401  .00006757  00000+0  51475-3 0  9997',
    '2 57912  43.0018 157.5807 0001420 272.5369  87.5310 15.02537576 31746'
]

# Create a TLE object
starlink30477 = sk.TLE.from_lines(tle_lines)

# We want the orbital state at April 9 2024, 12:00pm UTC
thetime = sk.time(2024, 4, 9, 12, 0, 0)

# The state is output in the "TEME" frame, which is an approximate inertial
# frame that does not include precession or nutation
# pTEME is geocentric position in meters
# vTEME is geocentric velocity in meters / second
# for now we will ignore the velocity
pTEME, _vTEME = sk.sgp4(starlink30477, thetime)

# Suppose we want current latitude, longitude, and altitude of satellite:
# we need to rotate into an Earth-fixed frame, the ITRF
# We use a "quaternion" to represent the rotation.  Quaternion rotations
# in the satkit toolbox can be represented as multiplications of a 3-vector
pITRF = sk.frametransform.qteme2itrf(thetime) * pTEME

# Now lets make a "ITRFCoord" object to extract geodetic coordinates
coord = sk.itrfcoord(pITRF)

# Get the latitude, longitude, and
# altitude (height above ellipsoid, or hae) of the satellite
print(coord)

## Plot Satellite Ground Track

The ground track is the projection of the satellite's orbit onto the Earth's surface. The sinusoidal pattern results from the satellite's inclined orbit combined with the Earth's rotation. The `mean_motion` field in the TLE gives the number of orbits per day, which we use to compute the total time span for 5 complete orbits.

In [ ]:
import ssl, certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
import satkit as sk
import numpy as np
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401

plt.style.use(["science", "no-latex"])
plt.rcParams.update({
    "mathtext.fontset": "stix",
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "STIXGeneral", "DejaVu Serif"],
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.formatter.use_mathtext": True,
    "svg.fonttype": "none",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "axes.prop_cycle": plt.cycler(color=[
        "#0077BB", "#EE7733", "#009988", "#CC3311",
        "#33BBEE", "#EE3377", "#BBBBBB",
    ]),
})
%config InlineBackend.figure_formats = ['svg']
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# The two-line element set
# Same satellite as above...
# The lines below were downloaded from https://www.celetrack.org
tle_lines = [
    '0 STARLINK-30477',
    '1 57912U 23146X   24099.49439401  .00006757  00000+0  51475-3 0  9997',
    '2 57912  43.0018 157.5807 0001420 272.5369  87.5310 15.02537576 31746'
]

# Create a TLE object
starlink30477 = sk.TLE.from_lines(tle_lines)

# We want the orbital state at April 9 2024, 12:00pm UTC
thetime = sk.time(2024, 4, 9, 12, 0, 0)

# plot for 5 orbits.  The mean motion in the TLE is number of orbits in a day
timearr = np.array([thetime + sk.duration(days=x) for x in np.linspace(0, 5/starlink30477.mean_motion, 1000)])

# Get position in the TEME frame
pTEME, _vTEME = sk.sgp4(starlink30477, timearr)
qarr = sk.frametransform.qteme2itrf(timearr)
pITRF = np.array([q * p for q, p in zip(qarr, pTEME)])
coord = [sk.itrfcoord(p) for p in pITRF]
lat, lon, alt = zip(*[(c.latitude_deg, c.longitude_deg, c.altitude) for c in coord])

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
# Break line at date line crossings
lon_arr, lat_arr = np.array(lon), np.array(lat)
breaks = np.where(np.abs(np.diff(lon_arr)) > 180)[0] + 1
lon_segs = np.split(lon_arr, breaks)
lat_segs = np.split(lat_arr, breaks)
for lo, la in zip(lon_segs, lat_segs):
    ax.plot(lo, la, linewidth=1, color="C0", transform=ccrs.PlateCarree())
ax.set_title("Ground Track")
ax.set_global()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot([t.as_datetime() for t in timearr], np.array(alt)/1e3)
ax.set_xlabel("Time")
ax.set_ylabel("Altitude (km)")
ax.set_title("Altitude vs Time")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()